In [1]:
import os
os.chdir('../')
%pwd

'/Users/kiranprasadjp/Documents/Pros/KidneyTumorCNNclassifier'

In [10]:
import dagshub
import mlflow

In [2]:
import dagshub
import mlflow
dagshub.init(repo_owner='captainkiran0158', repo_name='KidneyTumorCNNclassifier', mlflow=True)
print("Tracking URI:", mlflow.get_tracking_uri())


/Users/kiranprasadjp/Documents/Pros/KidneyTumorCNNclassifier/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as captainkiran0158

Initialized MLflow to track repo "captainkiran0158/KidneyTumorCNNclassifier"

Repository captainkiran0158/KidneyTumorCNNclassifier initialized!

Tracking URI: https://dagshub.com/captainkiran0158/KidneyTumorCNNclassifier.mlflow


In [2]:
import tensorflow as tf

In [4]:
model = tf.keras.models.load_model("artifacts/training/model.h5")

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [6]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories, save_json

In [11]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    
    def get_evaluation_config(self) -> EvaluationConfig:
        dagshub.init(repo_owner='captainkiran0158', repo_name='KidneyTumorCNNclassifier', mlflow=True)
        eval_config = EvaluationConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney-ct-scan-image",
            mlflow_uri=f"{mlflow.get_tracking_uri()}",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )
        return eval_config

In [12]:
import tensorflow as tf
from pathlib import Path
import mlflow.keras
from urllib.parse import urlparse

In [13]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    

    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme
        
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics(
                {"loss": self.score[0], "accuracy": self.score[1]}
            )
            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.keras.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.keras.log_model(self.model, "model")

    

In [14]:
try:
    config = ConfigurationManager()
    eval_config = config.get_evaluation_config()
    evaluation = Evaluation(eval_config)
    evaluation.evaluation()
    evaluation.log_into_mlflow()

except Exception as e:
   raise e

[2026-03-21 13:35:18,495: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-21 13:35:18,500: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-21 13:35:18,502: INFO: common: created directory at: artifacts]
[2026-03-21 13:35:18,798: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Accessing as captainkiran0158

[2026-03-21 13:35:18,803: INFO: helpers: Accessing as captainkiran0158]
[2026-03-21 13:35:19,096: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/repos/captainkiran0158/KidneyTumorCNNclassifier "HTTP/1.1 200 OK"]
[2026-03-21 13:35:19,337: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Initialized MLflow to track repo "captainkiran0158/KidneyTumorCNNclassifier"

[2026-03-21 13:35:19,343: INFO: helpers: Initialized MLflow to track repo "captainkiran0158/KidneyTumorCNNclassifier"]


...

9/9 [==============================] - 1s 113ms/step - loss: 2.9095e-04 - accuracy: 1.0000
[2026-03-21 13:35:21,279: INFO: common: json file saved at: scores.json]


2026/03/21 13:35:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 13:35:23 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


[2026-03-21 13:35:24,108: WARNING: save: Found untraced functions such as _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op, _jit_compiled_convolution_op while saving (showing 5 of 13). These functions will not be directly callable after loading.]
INFO:tensorflow:Assets written to: /var/folders/td/rr_6m89n4w93gmw6cv4n2rch0000gn/T/tmpww_za4b9/model/data/model/assets
[2026-03-21 13:35:24,487: INFO: builder_impl: Assets written to: /var/folders/td/rr_6m89n4w93gmw6cv4n2rch0000gn/T/tmpww_za4b9/model/data/model/assets]


2026/03/21 13:35:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'VGG16Model' already exists. Creating a new version of this model...
2026/03/21 13:36:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 4
Created version '4' of model 'VGG16Model'.


🏃 View run classy-flea-738 at: https://dagshub.com/captainkiran0158/KidneyTumorCNNclassifier.mlflow/#/experiments/0/runs/0c794481d2384486b8fcd4245fcc797b
🧪 View experiment at: https://dagshub.com/captainkiran0158/KidneyTumorCNNclassifier.mlflow/#/experiments/0
